# IBKR SOXX 期权偏斜前向来源探针

只读检查准确标的、常规到期、期权合约和一次盘后报价。没有历史回测、订单或期权交易。原始连接响应只在本卡内部文件中保留；合约 ID 不在用户结论中显示。

In [1]:
from pathlib import Path
import json,hashlib,datetime
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/ibkr_soxx_option_skew_forward_source_20261022"
pre=json.loads((card/"preflight_plan.json").read_text())
plan=json.loads((card/"sample_plan.json").read_text())
raw_path=card/"market_data_sample.json"
raw=json.loads(raw_path.read_text())
assert pre["frozen"] and plan["frozen"] and pre["budget"]["order_calls"]==plan["budget"]["order_calls"]==0
assert hashlib.sha256((card/"preflight_plan.json").read_bytes()).hexdigest()==plan["preflight_plan_sha256"]
assert len(raw["option_chain"]["contracts"])>0
assert raw["selected_contracts"]["strike"]=="495"
for side in ("call_snapshot","put_snapshot"):
    assert raw[side]["top-status"]["status"]=="FROZEN"
    assert raw[side]["option-midpoint-iv"]["isValid"] is False
    assert raw[side]["option-greeks"]=={}
result={"status":"needs_regular_hours_option_market_data_check","source":"Connected IBKR read-only market-data connector","underlying_quote_status":raw["underlying_snapshot"]["top-status"]["status"],"option_chain_rows":len(raw["option_chain"]["contracts"]),"regular_expiration":plan["expiration"],"selected_strike":raw["selected_contracts"]["strike"],"call_quote_status":raw["call_snapshot"]["top-status"]["status"],"put_quote_status":raw["put_snapshot"]["top-status"]["status"],"call_IV_valid":False,"put_IV_valid":False,"call_Delta_available":False,"put_Delta_available":False,"historical_skew_available":False,"forward_skew_qualified":False,"broker_order_calls":0,"model_fits":0,"accounts":0,"market_data_sample_sha256":hashlib.sha256(raw_path.read_bytes()).hexdigest(),"interpretation":"After-hours FROZEN option quotes with invalid IV and missing Greeks cannot establish an approximately 30-day 25-Delta skew at decision time. Regular-hours sample needed; no absence-of-entitlement inference from this after-hours observation."}
(card/"sample_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":"ibkr_soxx_option_skew_forward_source_20261022","status":result["status"],"decision":"source_preflight_only_no_skew_factor","result":"sample_result.json","recorder_status":"not_started_source_qualification_only","objective_achieved":False,"qualified_forward_days":0,"model_fits":0,"accounts":0,"artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["sources/*","analysis/*"]}}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\nIBKR 只读连接找到美股 SOXX 期权链和约 29 天后的常规到期，当前链有 "+str(result["option_chain_rows"])+" 个行权价。盘后标的状态为实时，但抽样看涨/看跌期权状态均为 FROZEN，IV 无效、Delta/Greeks 为空。\n\n这只说明盘后样本不能用于固定 25-Delta、约 30 天的偏斜；不能推断白天权限，也不证明历史来源。未创建订单、预警、模型或交易账户。下一步仅在正常交易时间对相同合约做一次预冻结的只读行情核验。\n")
print(result["status"],result["option_chain_rows"])

needs_regular_hours_option_market_data_check 168


## 中文结论

盘后样本缺有效 IV/Delta，不能形成固定 Delta 偏斜。详见同卡 `conclusion.md`。

## 常规时段只读资格观察

按 `regular_hours_plan.json` 三次固定快照读取同一标的和一对固定期权合约；此 notebook 仅分析保存的响应，不再发市场请求。


In [2]:
import math
from datetime import datetime, timezone
regular_plan=json.loads((card/'regular_hours_plan.json').read_text())
raw_path=card/'regular_hours_response_20261022.json'
raw=json.loads(raw_path.read_text())
assert raw['plan_sha256']==hashlib.sha256((card/'regular_hours_plan.json').read_bytes()).hexdigest()
assert raw['order_calls']==raw['alerts']==raw['watchlist_writes']==raw['account_reads']==raw['model_fits']==0
underlying,call,put=[raw['snapshots'][x]['parsed_response'] for x in ('underlying','call','put')]
def valid_quote(x):
    b=x['bid-ask']
    return all(math.isfinite(float(b[k])) for k in ('bid','ask')) and 0<float(b['bid'])<=float(b['ask'])
def valid_iv(x):
    v=x['option-midpoint-iv']
    return v.get('isValid') is True and math.isfinite(float(v['annualIv'])) and float(v['annualIv'])>0
def valid_delta(x,sign):
    d=float(x['option-greeks']['delta'])
    return math.isfinite(d) and 0<sign*d<1
realtime=all(x['top-status']['status']=='REALTIME' for x in (underlying,call,put))
entitlement_pass=bool(realtime and all(valid_quote(x) for x in (underlying,call,put)) and
    valid_iv(call) and valid_iv(put) and valid_delta(call,1) and valid_delta(put,-1))
observed=datetime.strptime(raw['observed_at_utc'],'%Y-%m-%d %H:%M:%S UTC').replace(tzinfo=timezone.utc)
assert datetime.fromisoformat(regular_plan['first_allowed_utc'])<=observed<=datetime.fromisoformat(regular_plan['last_allowed_utc'])
other=json.loads((card/'regular_hours_response.json').read_text())
duplicate=json.loads((card/'regular_hours_parallel_duplicate_20261022.json').read_text())
assert len(other['requests'])==duplicate['calls']==len(raw['snapshots'])==3
result={'status':'regular_hours_option_field_entitlement_passed' if entitlement_pass else 'needs_market_data',
    'regular_hours_plan_sha256':raw['plan_sha256'],
    'source_response_sha256':hashlib.sha256(raw_path.read_bytes()).hexdigest(),
    'canonical_response':'regular_hours_response_20261022.json',
    'additional_duplicate_receipts':['regular_hours_response.json','regular_hours_parallel_duplicate_20261022.json'],
    'additional_response_sha256':{name:hashlib.sha256((card/name).read_bytes()).hexdigest() for name in
        ('regular_hours_response.json','regular_hours_parallel_duplicate_20261022.json')},
    'snapshot_window':raw['snapshots']['underlying']['requested_at_utc']+' to '+raw['snapshots']['put']['received_at_utc'],
    'underlying_status':underlying['top-status']['status'],
    'call_status':call['top-status']['status'],'put_status':put['top-status']['status'],
    'call_bid_ask':[call['bid-ask']['bid'],call['bid-ask']['ask']],
    'put_bid_ask':[put['bid-ask']['bid'],put['bid-ask']['ask']],
    'call_IV_annual':call['option-midpoint-iv']['annualIv'],
    'put_IV_annual':put['option-midpoint-iv']['annualIv'],
    'call_Delta':call['option-greeks']['delta'],'put_Delta':put['option-greeks']['delta'],
    'sample_strike':regular_plan['strike'],'expiration':regular_plan['expiration'],
    'sample_is_25_delta':False,'option_quote_timestamps_provided':False,
    'historical_2011_2022_skew_available':False,'forward_25delta_skew_qualified':False,
    'frozen_plan_market_data_snapshots':3,'known_additional_duplicate_calls':6,
    'known_total_regular_hours_calls':9,'frozen_market_request_budget_exceeded':True,
    'independent_observations_from_duplicates':0,
    'chain_searches':0,'broker_order_calls':0,
    'account_reads':0,'model_fits':0,'native_accounts':0,
    'interpretation':'The frozen near-ATM 495 call/put proves regular-hours REALTIME fields, not a 25-Delta or constant-tenor feature. Concurrent tasks made six extra read-only snapshots; they are retained but do not add independent observations and exceeded the request budget. The connector omitted option quote timestamps and historical chain coverage. No Stage1 fit or account may use this as qualified input.'}
(card/'regular_hours_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary=json.loads((card/'summary.json').read_text())
summary.update(status=result['status'],decision='source_entitlement_passed_feature_not_qualified',
               result='regular_hours_result.json',regular_hours_result='regular_hours_result.json',
               historical_vendor_options='historical_vendor_options.json',qualified_forward_days=0)
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print({k:result[k] for k in ('status','call_Delta','put_Delta','call_IV_annual','put_IV_annual','forward_25delta_skew_qualified','known_total_regular_hours_calls')})


{'status': 'regular_hours_option_field_entitlement_passed', 'call_Delta': 0.4985732777931287, 'put_Delta': -0.5047797363994972, 'call_IV_annual': 0.4239029396640409, 'put_IV_annual': 0.4278202504097135, 'forward_25delta_skew_qualified': False, 'known_total_regular_hours_calls': 9}


## 最终结论

固定 495 行权价的看涨/看跌样本在常规时段返回 REALTIME、有效报价与 IV/Delta，但 Delta 约 ±0.50，连接未提供期权报价时间戳，且没有 2011–2022 期权链历史。此结果只证明字段可读，不构成 25Δ 近30天偏斜、Stage1 特征或经济超额。并行只读任务累计九张快照，超过原三张预算，重复观察不算独立前向；四行权价后续预检未执行。历史资料可行性与未核实边界见 `historical_vendor_options.json`；无付费采购。
